<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/1_evolution_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Evolution Strategyによる非制約連続最適化

Evolution Strategy (ES) は，進化計算と呼ばれる"生物の進化に着想を得た"最適化アルゴリズムの一種である．現在，Covariance Matrix Adaptation Evolution Strategy (CMA-ES) と呼ばれるESの一種が，最も優れたアルゴリズムの一つとして広く用いられている．

ここでは，Step-Size Adaptive ESと呼ばれるシンプルなアルゴリズムを例にとり，ESの動作原理を理解する．

<!-- EN -->
# Unconstrained Continuous Optimization via Evolution Strategy

<!-- EN -->
Evolution Strategy (ES) is a type of optimization algorithm belonging to a family known as evolutionary computation, which draws inspiration from biological evolution. Currently, a variant of ES called Covariance Matrix Adaptation Evolution Strategy (CMA-ES) is widely used as one of the best-performing algorithms.

Here, we use a simple algorithm called Step-Size Adaptive ES as an example to understand the operating principles of ES.

## ステップサイズの更新を行わないES

ここでは，ESのほとんどの機能を取り除いた骨格だけの場合に，どのような振る舞いをするのかを確認する．その上で，この先のノートブックにおいて，現在CMA-ESに実装されている各コンポーネントの役割を一つずつ確認していく．

以下では，次の手続きを繰り返すだけのランダムサーチを実装している．

1. 平均ベクトル$m$，ステップサイズ$\sigma$を初期化
2. 正規分布$\mathcal{N}(m, \sigma^2 I)$に従って，$\lambda$個の独立な解 $x_i = m + \sigma \mathcal{N}(0, I)$ (for $i = 1,\dots, \lambda$) を生成
3. それぞれの解の目的関数値 $f(x_i)$ を（並列に）評価
4. 目的関数の昇順に解をソート．$x_{i:\lambda}$ を $i$番目に目的関数値の小さな解とする．
5. 平均ベクトルを以下のように更新
$$
m \leftarrow m + \sum_{i=1}^{\lambda} w_{i} (x_{i:\lambda} - m) = \sum_{i=1}^{\lambda} w_{i} x_{i:\lambda}
$$
6. ステップ2に戻る

ここで，$w_1 \leq \dots \leq w_\lambda$ は解のランキング毎に与えられる重みであり，以下では簡単のため$\mu = \lfloor \lambda / 4 \rfloor$とし（上位解の選択数），$w_1 = \dots = w_\mu = 1/\mu$，$w_{\mu+1} = \dots = w_{\lambda} = 0$ とする．最新のCMA-ESでは重みに傾斜をつけているが，大幅に探索性能が変わるわけでは無いため，簡単のためこのような重みを利用する．

以上のように，このアルゴリズムは，複数の解を正規分布から生成し，上位$\mu$個の解の平均値を次の正規分布の平均ベクトルとする，ということを繰り返すだけのランダムサーチである．

<!-- EN -->
## ES Without Step-Size Update

<!-- EN -->
Here, we examine the behavior of the ES stripped down to just its skeleton, with most features removed. Building on this, in subsequent notebooks we will verify the role of each component currently implemented in CMA-ES, one by one.

The following implements a random search that simply repeats the following procedure:

1. Initialize the mean vector $m$ and step size $\sigma$
2. Sample $\lambda$ independent solutions $x_i = m + \sigma \mathcal{N}(0, I)$ (for $i = 1,\dots, \lambda$) from the normal distribution $\mathcal{N}(m, \sigma^2 I)$
3. Evaluate the objective function value $f(x_i)$ for each solution (in parallel)
4. Sort solutions in ascending order of objective function value. Let $x_{i:\lambda}$ denote the solution with the $i$-th smallest objective function value.
5. Update the mean vector as follows:
$$
m \leftarrow m + \sum_{i=1}^{\lambda} w_{i} (x_{i:\lambda} - m) = \sum_{i=1}^{\lambda} w_{i} x_{i:\lambda}
$$
6. Return to step 2

Here, $w_1 \leq \dots \leq w_\lambda$ are weights assigned to each ranking of the solutions. For simplicity, we set $\mu = \lfloor \lambda / 4 \rfloor$ (the number of selected top solutions), with $w_1 = \dots = w_\mu = 1/\mu$ and $w_{\mu+1} = \dots = w_{\lambda} = 0$. The latest CMA-ES uses graduated weights, but since this does not substantially change the search performance, we use these uniform weights for simplicity.

As described above, this algorithm is a random search that repeatedly generates multiple solutions from a normal distribution and sets the average of the top $\mu$ solutions as the mean vector of the next normal distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class ES(object):
    """Evolution Strategy with Constant Step-Size"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """Constructor

        Parameters
        ----------
        func : callable
            objective function (to be minimized)
        init_mean : ndarray (1D)
            initial mean vector
        init_sigma : float
            initial step-size
        nsample : int
            population size, sample size
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # search space dimension
        self.arx = np.zeros((nsample, self.N)) * np.nan # candidate solutions
        self.arf = np.zeros(nsample) * np.nan           # fitness of candidates

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # weights, sum to 1.

    def sample(self):
        """generate candidate solutions"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape)

    def evaluate(self):
        """evaluate candidate solutions"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_mean(self):
        """update the mean vector"""
        idx = np.argsort(self.arf)  # idx[i] is the index of the i-th best candidate
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))


In [ ]:
def sphere(x):
    """Sphere fucntion. Opt = (0,...,0)"""
    return np.linalg.norm(x)

####ESを用いてSphere関数を最適化する．

<!-- EN -->
#### Optimize the Sphere function using ES.

In [ ]:
es = ES(func=sphere,
        init_mean=np.ones(10),
        init_sigma=0.001,
        nsample=10)

maxiter = 5000
fbest = np.zeros(maxiter) * np.nan
fmean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(fbest.shape[0]):
    es.sample()
    es.evaluate()
    es.update_mean()
    fbest[i] = es.arf.min()
    fmean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N

####結果をプロットする．

<!-- EN -->
#### Plot the results.

In [ ]:
plt.semilogy(fbest, '-r', label='f(best)')
plt.semilogy(fmean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend()

##考察

* init_mean や init_sigma を変えて実験してみよう (適宜maxiterを変更すること)
* 振る舞いを三つの状況(序盤の遅い探索，中盤の速い探索，終盤の停滞)に分けて，それぞれmeanとsigmaがどのような関係にあるのか考えよう
* 中盤の速い探索が続くように，ステップサイズの更新を実装しよう．

<!-- EN -->
## Discussion

<!-- EN -->
* Try experimenting by varying `init_mean` and `init_sigma` (adjust `maxiter` as appropriate)
* Divide the behavior into three phases (slow exploration in the early phase, fast exploration in the middle phase, and stagnation in the late phase), and consider the relationship between `mean` and `sigma` in each phase
* Implement a step-size update so that the fast exploration of the middle phase continues.

In [ ]:
es = ES(func=sphere,
        init_mean=np.ones(10),
        init_sigma=0.001,
        nsample=10)

maxiter = 100
fbest = np.zeros(maxiter) * np.nan
fmean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(fbest.shape[0]):
    es.sample()
    es.evaluate()
    es.update_mean()
    # start: update the step-size ---
    old_sigma = es.sigma

    es.sigma = sphere(es.mean) / es.N
    # end: update the step-size
    fbest[i] = es.arf.min()
    fmean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N